# PDANA Exploratory Data Analysis
## Anonymised VWGA dealer-level data

This notebook loads the anonymised Excel workbook from the fixed PDANA POE DATA folder.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


## 1. Load the anonymised analytical workbook


In [ ]:
DATA_DIR = Path(r"C:\Users\dzauakr\OneDrive - Volkswagen AG\Desktop\PGDip Data Analytics 2026\Programming for Data Analytics PDAN8411\PDANA POE DATA")
DATA_FILE = DATA_DIR / "PDANA_Anonymised_EDA.xlsx"
NOTEBOOK_FILE = DATA_DIR / "PDANA_Anonymised_EDA_Fixed.ipynb"

print("Expected folder:", DATA_DIR)
print("Excel path:", DATA_FILE)
print("Excel exists:", DATA_FILE.exists())

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {DATA_DIR}")
if not DATA_FILE.exists():
    available = [p.name for p in DATA_DIR.glob("*.xlsx")]
    raise FileNotFoundError(
        f"Expected Excel file not found: {DATA_FILE}\n"
        f"Excel files found in the folder: {available}"
    )

# List sheets first so a sheet-name mismatch gives a useful message.
xls = pd.ExcelFile(DATA_FILE, engine="openpyxl")
print("Sheets found:", xls.sheet_names)

required_sheet = "Anonymised Data"
if required_sheet not in xls.sheet_names:
    raise ValueError(
        f"Sheet '{required_sheet}' was not found. Available sheets: {xls.sheet_names}"
    )

df = pd.read_excel(DATA_FILE, sheet_name=required_sheet, engine="openpyxl")
display(df.head())


## 2. Structure and data-quality review


In [ ]:
print("Shape:", df.shape)
print(df.dtypes)
display(df.isna().sum().to_frame("Missing values"))
print("Duplicate rows:", df.duplicated().sum())
print("Unique dealer aliases:", df["Dealer Alias"].nunique())
print("Score range:", df["Satisfaction Score"].min(), "to", df["Satisfaction Score"].max())


In [ ]:
quality_checks = pd.Series({
    "Rows loaded": len(df),
    "Missing aliases": df["Dealer Alias"].isna().sum(),
    "Missing scores": df["Satisfaction Score"].isna().sum(),
    "Scores outside 1-5": (~df["Satisfaction Score"].between(1, 5)).sum(),
    "Ranking shifts reported": df["Ranking Shift"].notna().sum(),
    "Unique aliases": df["Dealer Alias"].nunique()
})
display(quality_checks.to_frame("Result"))


## 3. Overall descriptive statistics


In [ ]:
overall_stats = df["Satisfaction Score"].describe().to_frame("Satisfaction Score")
overall_stats.loc["median"] = df["Satisfaction Score"].median()
display(overall_stats)


In [ ]:
band_order = ["High (>=4.5)", "Moderate (4.0-4.49)", "Low (<4.0)"]
band_counts = df["Score Band"].value_counts().reindex(band_order).fillna(0).astype(int)
display(band_counts.to_frame("Count"))
band_counts.plot(kind="bar", color=["#00A6A6", "#F2B134", "#C94C4C"], title="Satisfaction score bands")
plt.xlabel("Score band")
plt.ylabel("Number of observations")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 4. Comparison by customer-journey segment


In [ ]:
segment_summary = (
    df.groupby("Customer Journey Segment")["Satisfaction Score"]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .round(3)
      .sort_values("mean", ascending=False)
)
display(segment_summary)


In [ ]:
ax = segment_summary["mean"].sort_values().plot(
    kind="barh", color="#007A8A", title="Mean satisfaction score by segment"
)
ax.set_xlabel("Mean satisfaction score")
ax.set_ylabel("")
ax.set_xlim(3.0, 5.0)
plt.tight_layout()
plt.show()


In [ ]:
segments = list(df["Customer Journey Segment"].dropna().unique())
data = [df.loc[df["Customer Journey Segment"].eq(s), "Satisfaction Score"] for s in segments]
plt.boxplot(data, tick_labels=segments, showmeans=True)
plt.title("Distribution of satisfaction scores by segment")
plt.ylabel("Satisfaction score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 5. Lowest and highest anonymised observations


In [ ]:
cols = ["Dealer Alias", "Customer Journey Segment", "Satisfaction Score", "Ranking Shift"]
print("Lowest 10 observations")
display(df.nsmallest(10, "Satisfaction Score")[cols])
print("Highest 10 observations")
display(df.nlargest(10, "Satisfaction Score")[cols])


## 6. Ranking-shift analysis


In [ ]:
shifts = df.dropna(subset=["Ranking Shift"]).copy()
print("Reported ranking shifts:", len(shifts))
display(shifts["Ranking Shift"].describe().to_frame())
if not shifts.empty:
    shifts.sort_values("Ranking Shift").plot(
        x="Dealer Alias", y="Ranking Shift", kind="bar", color="#5B6770",
        legend=False, title="Reported ranking shifts"
    )
    plt.axhline(0, color="black", linewidth=0.8)
    plt.ylabel("Ranking shift")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.show()


## 7. Repeated aliases across segments


In [ ]:
repeated = (
    df.groupby("Dealer Alias")
      .agg(Observations=("Dealer Alias", "size"),
           Segments=("Customer Journey Segment", "nunique"),
           Mean_Score=("Satisfaction Score", "mean"),
           Min_Score=("Satisfaction Score", "min"),
           Max_Score=("Satisfaction Score", "max"))
      .query("Observations > 1")
      .sort_values(["Segments", "Observations"], ascending=False)
      .round(3)
)
display(repeated)


## 8. Conditional text EDA
Runs only if an anonymised `Comment` column is available.


In [ ]:
if "Comment" in df.columns:
    import re
    from collections import Counter
    comments = df["Comment"].dropna().astype(str)
    display(comments.str.split().str.len().describe().to_frame("Comment word count"))
    stopwords = {"the","and","a","to","of","is","was","for","in","my","it","that","with","on","i","very","be","had"}
    tokens = []
    for text in comments.str.lower():
        tokens.extend([w for w in re.findall(r"[a-z']+", text) if len(w) > 2 and w not in stopwords])
    display(pd.DataFrame(Counter(tokens).most_common(20), columns=["Term", "Frequency"]))
else:
    print("No Comment column is present. Numeric dealer-level EDA completed; text EDA is deferred.")
